In [27]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report,accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
import numpy as np
from sklearn.compose import ColumnTransformer

In [28]:
df=pd.read_csv('insurance.csv')

In [29]:
df.sample(5)

,age,weight,height,income_lpa,smoker,city,occupation,insurance_premium_category
11,25,77.2,1.56,10.899387,True,Pune,government_job,Low
60,41,101.3,1.81,49.940000,True,Jalandhar,unemployed,High
63,47,71.3,1.82,41.660000,True,Gaya,business_owner,High
43,72,85.7,1.71,1.560000,False,Chennai,retired,Medium
12,42,95.2,1.78,17.580000,True,Chandigarh,freelancer,High


In [30]:
df_feat=df.copy()

In [31]:
df_feat["bmi"]=df_feat["weight"]/ (df_feat["height"]**2)

In [32]:
def age_group(age):
  if age<25:
    return "young"
  elif age<45:
    return "adult"
  elif age<65:
    return "middle age"
  return "senoir"

In [33]:
df_feat["age_group"] = df_feat["age"].apply(age_group)

In [34]:
def lifestyle_risk(row):
  if row["smoker"] and row["bmi"]>30:
    return "high"
  elif row["smoker"] or row["bmi"]<27:
    return "medium"
  else:
    return "low"

In [35]:
df_feat["lifestyle_risk"]=df_feat.apply(lifestyle_risk,axis=1)

In [36]:
tier_1_cities = ["Mumbai", "Delhi", "Bangalore", "Chennai", "Kolkata", "Hyderabad", "Pune"]
tier_2_cities = [
    "Jaipur", "Chandigarh", "Indore", "Lucknow", "Patna", "Ranchi", "Visakhapatnam", "Coimbatore",
    "Bhopal", "Nagpur", "Vadodara", "Surat", "Rajkot", "Jodhpur", "Raipur", "Amritsar", "Varanasi",
    "Agra", "Dehradun", "Mysore", "Jabalpur", "Guwahati", "Thiruvananthapuram", "Ludhiana", "Nashik",
    "Allahabad", "Udaipur", "Aurangabad", "Hubli", "Belgaum", "Salem", "Vijayawada", "Tiruchirappalli",
    "Bhavnagar", "Gwalior", "Dhanbad", "Bareilly", "Aligarh", "Gaya", "Kozhikode", "Warangal",
    "Kolhapur", "Bilaspur", "Jalandhar", "Noida", "Guntur", "Asansol", "Siliguri"
]

In [37]:
def city_tier(city):
    if city in tier_1_cities:
        return 1
    elif city in tier_2_cities:
        return 2
    else:
        return 3


In [38]:
df_feat["city_tier"] = df_feat["city"].apply(city_tier)

In [39]:
df_feat.drop(columns=['age', 'weight', 'height', 'smoker', 'city'])[['income_lpa', 'occupation', 'bmi','age_group', 'lifestyle_risk', 'city_tier', 'insurance_premium_category']].sample(5)

,income_lpa,occupation,bmi,age_group,lifestyle_risk,city_tier,insurance_premium_category
75,45.070000,unemployed,20.577355,middle age,medium,1,Low
94,10.542289,government_job,33.266002,middle age,low,1,Low
66,3.230000,student,25.275899,young,medium,2,Low
20,30.650000,business_owner,17.005113,adult,medium,2,Medium
72,3.080000,retired,35.499527,senoir,low,2,High


In [40]:
X = df_feat[["bmi", "age_group", "lifestyle_risk", "city_tier", "income_lpa", "occupation"]]
y = df_feat["insurance_premium_category"]

In [41]:
categorical_features = ["age_group", "lifestyle_risk", "occupation", "city_tier"]
numeric_features = ["bmi", "income_lpa"]

In [42]:
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(), categorical_features),
        ("num", "passthrough", numeric_features)
    ]
)

In [43]:
pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(random_state=42))
])

In [44]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=1)
pipeline.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('cat', OneHotEncoder(),
                                                  ['age_group',
                                                   'lifestyle_risk',
                                                   'occupation', 'city_tier']),
                                                 ('num', 'passthrough',
                                                  ['bmi', 'income_lpa'])])),
                ('classifier', RandomForestClassifier(random_state=42))])

In [45]:
y_pred = pipeline.predict(X_test)
accuracy_score(y_test, y_pred)

0.8

In [46]:
X_test.sample(5)

,bmi,age_group,lifestyle_risk,city_tier,income_lpa,occupation
93,23.199416,young,medium,2,1.28,student
80,34.350461,middle age,low,2,50.00,unemployed
36,21.713266,middle age,medium,1,0.53,retired
81,31.866055,adult,high,2,22.19,freelancer
10,22.949982,adult,medium,1,32.78,business_owner


In [47]:

import pickle

# Save the trained pipeline using pickle
pickle_model_path = "model.pkl"
with open(pickle_model_path, "wb") as f:
    pickle.dump(pipeline, f)
